# Lag e Target da `dataset_pulito3.csv`

Questo notebook legge `dataset_pulito3.csv` e costruisce solo:
- lag storici per cella geografica
- target futuri per previsione incendi

Output finale: `dataset_pulito3_lag_target_final.csv`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

INPUT_FILE = Path("dataset_pulito3.csv")
OUTPUT_FILE = Path("dataset_pulito3_lag_target_final.csv")

CELL_COLUMNS = ["lat_cell", "lon_cell"]
DATE_CANDIDATES = ["chunk_start", "chunk_end", "acq_date", "date"]
LAG_COLUMNS = [
    "fire_count_last_1d",
    "fire_count_last_3d",
    "fire_count_last_7d",
    "frp_mean_last_7d",
    "days_since_last_fire",
]
TARGET_COLUMNS = [
    "fire_next_1d",
    "fire_next_3d",
    "fire_next_7d",
    "fire_count_next_7d",
]

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File non trovato: {INPUT_FILE.resolve()}")

source_df = pd.read_csv(INPUT_FILE)

required_columns = set(CELL_COLUMNS + ["frp"])
missing_columns = required_columns.difference(source_df.columns)
if missing_columns:
    raise ValueError(f"Colonne mancanti nel CSV: {sorted(missing_columns)}")

available_date_columns = [column for column in DATE_CANDIDATES if column in source_df.columns]
if not available_date_columns:
    raise ValueError(
        "Serve almeno una colonna data tra: " + ", ".join(DATE_CANDIDATES)
    )

print(f"Input file: {INPUT_FILE.resolve()}")
print(f"Righe lette: {len(source_df):,}")
print(f"Colonne data disponibili: {available_date_columns}")

In [ ]:
date = pd.Series(pd.NaT, index=source_df.index, dtype="datetime64[ns]")
for column in DATE_CANDIDATES:
    if column in source_df.columns:
        date = date.fillna(pd.to_datetime(source_df[column], errors="coerce"))

source_df = source_df.assign(
    date=date.dt.normalize(),
    frp=pd.to_numeric(source_df["frp"], errors="coerce"),
)

daily_df = (
    source_df.drop_duplicates()
    .dropna(subset=CELL_COLUMNS + ["date"])
    .groupby(CELL_COLUMNS + ["date"], as_index=False)
    .agg(
        fire_count=("frp", "size"),
        daily_frp_mean=("frp", "mean"),
        source_rows=("frp", "size"),
    )
    .sort_values(CELL_COLUMNS + ["date"], ignore_index=True)
)

full_grid = (
    daily_df[CELL_COLUMNS]
    .drop_duplicates()
    .merge(
        pd.DataFrame({"date": pd.date_range(daily_df["date"].min(), daily_df["date"].max(), freq="D")}),
        how="cross",
    )
)

lag_df = (
    full_grid.merge(daily_df, on=CELL_COLUMNS + ["date"], how="left", validate="one_to_one")
    .sort_values(CELL_COLUMNS + ["date"], ignore_index=True)
)

lag_df["fire_count"] = lag_df["fire_count"].fillna(0).astype("int32")
lag_df["source_rows"] = lag_df["source_rows"].fillna(0).astype("int32")
lag_df["month"] = lag_df["date"].dt.month.astype("Int8")
lag_df["day_of_year"] = lag_df["date"].dt.dayofyear.astype("Int16")
lag_df["week"] = lag_df["date"].dt.isocalendar().week.astype("Int16")
lag_df["sin_doy"] = np.sin(2 * np.pi * lag_df["day_of_year"].astype(float) / 365.0)
lag_df["cos_doy"] = np.cos(2 * np.pi * lag_df["day_of_year"].astype(float) / 365.0)

g = lag_df.groupby(CELL_COLUMNS, sort=False)
history_len = g.cumcount()
past_fire = pd.concat([g["fire_count"].shift(i).rename(i) for i in range(1, 8)], axis=1).astype("Float64")
past_frp = pd.concat([g["daily_frp_mean"].shift(i).rename(i) for i in range(1, 8)], axis=1).astype("Float64")

last_fire_date = lag_df["date"].where(lag_df["fire_count"] > 0)
cell_keys = [lag_df["lat_cell"], lag_df["lon_cell"]]
prev_fire_date = last_fire_date.groupby(cell_keys, sort=False).ffill().groupby(cell_keys, sort=False).shift(1)

lag_df["fire_count_last_1d"] = past_fire[1].round().astype("Int64")
lag_df["fire_count_last_3d"] = past_fire[[1, 2, 3]].sum(axis=1, min_count=3).round().astype("Int64")
lag_df["fire_count_last_7d"] = past_fire.sum(axis=1, min_count=7).round().astype("Int64")
lag_df["frp_mean_last_7d"] = past_frp.mean(axis=1).astype("Float64")
lag_df["days_since_last_fire"] = (lag_df["date"] - prev_fire_date).dt.days.astype("Int64")

lag_df.loc[history_len < 1, "fire_count_last_1d"] = pd.NA
lag_df.loc[history_len < 3, "fire_count_last_3d"] = pd.NA
lag_df.loc[history_len < 7, ["fire_count_last_7d", "frp_mean_last_7d"]] = pd.NA

print("Missing values in lag columns:")
print(lag_df[LAG_COLUMNS].isna().sum())
print(f"Shape dopo creazione lag: {lag_df.shape}")

lag_df.head()

In [ ]:
def to_binary(series):
    out = pd.Series(pd.NA, index=series.index, dtype="Int8")
    mask = series.notna()
    out.loc[mask] = series.loc[mask].gt(0).astype("int8")
    return out


g = lag_df.groupby(CELL_COLUMNS, sort=False)
future_fire = pd.concat([g["fire_count"].shift(-i).rename(i) for i in range(1, 8)], axis=1).astype("Float64")

next_1d = future_fire[1]
next_3d = future_fire[[1, 2, 3]].sum(axis=1, min_count=3)
next_7d = future_fire.sum(axis=1, min_count=7)

final_temporal_df = lag_df.copy()
final_temporal_df["fire_next_1d"] = to_binary(next_1d)
final_temporal_df["fire_next_3d"] = to_binary(next_3d)
final_temporal_df["fire_next_7d"] = to_binary(next_7d)
final_temporal_df["fire_count_next_7d"] = next_7d.round().astype("Int64")

final_temporal_df = final_temporal_df[
    [
        "date", "lat_cell", "lon_cell", "fire_count", "daily_frp_mean", "source_rows",
        "month", "day_of_year", "week", "sin_doy", "cos_doy",
        "fire_count_last_1d", "fire_count_last_3d", "fire_count_last_7d",
        "frp_mean_last_7d", "days_since_last_fire",
        "fire_next_1d", "fire_next_3d", "fire_next_7d", "fire_count_next_7d",
    ]
].copy()

final_temporal_df.to_csv(OUTPUT_FILE, index=False)

print(f"Final dataset shape: {final_temporal_df.shape}")
print(f"Saved file: {OUTPUT_FILE.resolve()}")
print("Missing values in target columns:")
print(final_temporal_df[TARGET_COLUMNS].isna().sum())

final_temporal_df.head()